# Electric Production - Time Series Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('Data/Electric_Production.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
print('Data types:')
print(df.dtypes)
print(f'\nMissing values:\n{df.isnull().sum()}')

## 2. Parse Dates & Set Index

In [ ]:
df['DATE'] = pd.to_datetime(df['DATE'], format='%m-%d-%Y')
df = df.sort_values('DATE').set_index('DATE')
df = df.asfreq('MS')  # Monthly start frequency
print(f'Date range: {df.index.min()} → {df.index.max()}')
df.head()

## 3. Data Quality Checks

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Duplicate Dates ===')
print(df.index.duplicated().sum())

print('\n=== Basic Statistics ===')
df.describe()

## 3.5 Outlier Detection

In [ ]:
from scipy import stats

# --- Z-Score Method ---
z_scores = np.abs(stats.zscore(df['Value'].dropna()))
zscore_outliers = df['Value'][z_scores > 3]
print(f'Z-Score outliers (threshold > 3): {len(zscore_outliers)}')

# --- IQR Method ---
Q1 = df['Value'].quantile(0.25)
Q3 = df['Value'].quantile(0.75)
IQR = Q3 - Q1
iqr_mask = (df['Value'] < Q1 - 1.5 * IQR) | (df['Value'] > Q3 + 1.5 * IQR)
iqr_outliers = df['Value'][iqr_mask]
print(f'IQR outliers (1.5 * IQR)        : {len(iqr_outliers)}')

# Combine both for visualization
all_outlier_idx = zscore_outliers.index.union(iqr_outliers.index)

# Plot
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Value'], color='steelblue', linewidth=1.5, label='Original')
plt.scatter(all_outlier_idx, df.loc[all_outlier_idx, 'Value'],
            color='red', zorder=5, s=60, label='Outliers')
plt.title('Outlier Detection (Z-Score & IQR)', fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Production Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# List outlier dates and values
print('\nOutlier dates and values:')
print(df.loc[all_outlier_idx, 'Value'].to_string())

## 3.6 Distribution Analysis

In [ ]:
from scipy.stats import shapiro, probplot

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Value'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Production Values', fontweight='bold')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Q-Q Plot
probplot(df['Value'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (Normality Check)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Shapiro-Wilk Test
stat, p_value = shapiro(df['Value'].dropna())
print(f'Shapiro-Wilk Test:')
print(f'  Statistic : {stat:.4f}')
print(f'  p-value   : {p_value:.4f}')
print(f'  Result    : {"NORMAL distribution" if p_value > 0.05 else "NOT normal distribution"} (p-threshold = 0.05)')

## 4. Visualize the Time Series

In [ ]:
plt.plot(df.index, df['Value'], linewidth=1.5, color='steelblue')
plt.title('Electric Production (1985 - 2018)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Production Value')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Rolling Statistics (Trend & Variability)

In [ ]:
rolling_mean = df['Value'].rolling(window=12).mean()
rolling_std  = df['Value'].rolling(window=12).std()

plt.plot(df['Value'],       label='Original',       alpha=0.5)
plt.plot(rolling_mean,      label='12-Month Mean',  linewidth=2)
plt.plot(rolling_std,       label='12-Month Std',   linewidth=2, linestyle='--')
plt.title('Rolling Mean & Standard Deviation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.5 Lag Plot Analysis

In [ ]:
from pandas.plotting import lag_plot

lags = [1, 6, 12, 24]
lag_labels = ['Lag 1', 'Lag 6', 'Lag 12 (Seasonal)', 'Lag 24']
colors = ['steelblue', 'darkorange', 'green', 'purple']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (lag, label, color) in enumerate(zip(lags, lag_labels, colors)):
    axes[i].scatter(df['Value'][:-lag].values, df['Value'][lag:].values,
                    color=color, alpha=0.5, s=20)
    axes[i].set_title(label, fontweight='bold')
    axes[i].set_xlabel('y(t)')
    axes[i].set_ylabel(f'y(t+{lag})')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Lag Plot Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Stationarity Check — ADF & KPSS Tests

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def adf_test(series, label='Series'):
    result = adfuller(series.dropna())
    print(f'--- ADF Test: {label} ---')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    print(f'  Critical Values: { {k: f"{v:.4f}" for k, v in result[4].items()} }')
    print(f'  Result        : {"STATIONARY" if result[1] < 0.05 else "NON-STATIONARY"}\n')

def kpss_test(series, label='Series'):
    result = kpss(series.dropna(), regression='c', nlags='auto')
    print(f'--- KPSS Test: {label} ---')
    print(f'  KPSS Statistic: {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    print(f'  Critical Values: { {k: f"{v:.4f}" for k, v in result[3].items()} }')
    print(f'  Result        : {"NON-STATIONARY" if result[1] < 0.05 else "STATIONARY"}\n')

adf_test(df['Value'],  'Original')
kpss_test(df['Value'], 'Original')

## 7. Make Stationary (if needed)

In [ ]:
# Step 1: First-order differencing to remove trend
df['Value_diff1'] = df['Value'].diff()

# Step 2: Seasonal differencing (lag=12) to remove seasonality
df['Value_diff1_seasonal'] = df['Value_diff1'].diff(12)

print('After 1st-order differencing:')
adf_test(df['Value_diff1'], '1st Diff')
kpss_test(df['Value_diff1'], '1st Diff')

print('After 1st-order + Seasonal differencing:')
adf_test(df['Value_diff1_seasonal'], '1st + Seasonal Diff')
kpss_test(df['Value_diff1_seasonal'], '1st + Seasonal Diff')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(df['Value'], color='steelblue')
axes[0].set_title('Original Series')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['Value_diff1'], color='darkorange')
axes[1].set_title('After 1st-Order Differencing')
axes[1].grid(True, alpha=0.3)

axes[2].plot(df['Value_diff1_seasonal'], color='green')
axes[2].set_title('After 1st-Order + Seasonal Differencing (lag=12)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7.5 ACF & PACF Analysis

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

series_list = [
    (df['Value'],                  'Original Series',                    'steelblue'),
    (df['Value_diff1'],            '1st-Order Differenced',              'darkorange'),
    (df['Value_diff1_seasonal'],   '1st-Order + Seasonal Differenced',   'green'),
]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for i, (series, label, color) in enumerate(series_list):
    plot_acf(series.dropna(),  lags=40, ax=axes[i, 0], color=color, title=f'ACF — {label}')
    plot_pacf(series.dropna(), lags=40, ax=axes[i, 1], color=color, title=f'PACF — {label}')
    axes[i, 0].grid(True, alpha=0.3)
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Time Series Decomposition — Trend, Seasonality, Cyclicity & Irregularity

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Multiplicative decomposition (better when seasonal amplitude grows with trend)
decomp = seasonal_decompose(df['Value'], model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 14))

axes[0].plot(decomp.trend,    color='steelblue')
axes[0].set_title('Trend Component', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(decomp.seasonal, color='darkorange')
axes[1].set_title('Seasonal Component (period=12 months)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[2].plot(decomp.resid,    color='green')
axes[2].axhline(1, color='red', linestyle='--', linewidth=1)
axes[2].set_title('Residual / Irregular Component', fontweight='bold')
axes[2].grid(True, alpha=0.3)

axes[3].plot(df['Value'],     color='gray', alpha=0.7)
axes[3].plot(decomp.trend,    color='steelblue', linewidth=2, label='Trend')
axes[3].set_title('Original + Trend Overlay', fontweight='bold')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Cyclicity Analysis — HP Filter

In [ ]:
from statsmodels.tsa.filters.hp_filter import hpfilter

# lambda=1600 is standard for monthly data
cycle, trend_hp = hpfilter(df['Value'].dropna(), lamb=129600)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df['Value'].dropna().index, trend_hp, color='steelblue', linewidth=2)
axes[0].set_title('Long-Term Trend (HP Filter)', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['Value'].dropna().index, cycle, color='crimson', linewidth=1.5)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Cyclical Component (HP Filter)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Seasonal Pattern — Monthly Boxplot

In [ ]:
import calendar

df['Month'] = df.index.month
monthly_data = [df[df['Month'] == m]['Value'].values for m in range(1, 13)]

plt.figure(figsize=(14, 5))
plt.boxplot(monthly_data, labels=calendar.month_abbr[1:], patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
plt.title('Seasonal Pattern — Monthly Distribution', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Production Value')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Save Preprocessed Data

In [ ]:
df.to_csv('Data/Electric_Production_Preprocessed.csv')
print('Saved → Data/Electric_Production_Preprocessed.csv')
df.tail()